# Kaggle Runner: `vit_s16_imagenet_covidqu_syn`

Use this notebook on Kaggle GPU after uploading/adding the improved DCGAN synthetic dataset as a Kaggle Dataset. Pretraining and fine-tuning are split into separate cells.


## 1. Setup Repository


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, csv, json
import pandas as pd

print('Kaggle input exists:', Path('/kaggle/input').exists())
print('Kaggle working exists:', Path('/kaggle/working').exists())
subprocess.run(['nvidia-smi'], check=False)

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/kaggle/working/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    os.chdir(REPO_ROOT)
    subprocess.run(['git', 'pull'], check=True)
else:
    os.chdir('/kaggle/working')
    subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir(REPO_ROOT)

print('REPO_ROOT:', REPO_ROOT)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


## 2. Install Dependencies


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'scikit-learn', 'matplotlib', 'pandas', 'Pillow', 'pyyaml'], check=True)
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## 3. Edit Kaggle Dataset Paths


In [ ]:
EXPERIMENT_ID = 'vit_s16_imagenet_covidqu_syn'
CONFIG_PATH = Path('configs/experiments/vit_s16/imagenet_covidqu_syn.yaml')
OUTPUT_ROOT = Path('/kaggle/working/results/experiments_rerun_dcgan128')
OUT = OUTPUT_ROOT / EXPERIMENT_ID

# Edit these paths to match the Kaggle Dataset you add to the notebook.
# The dataset should contain data/processed/labelled_4232, data/processed/unlabelled_16934,
# and the improved synthetic folder, preferably data/processed/synthetic_dcgan_stabilized.
DATA_ROOT = Path('/kaggle/input/medcls-cvproject/data')
LABELLED_SOURCE = DATA_ROOT / 'processed/labelled_4232'
UNLABELLED_SOURCE = DATA_ROOT / 'processed/unlabelled_16934'
SYNTHETIC_SOURCE = DATA_ROOT / 'processed/synthetic_dcgan_stabilized'
MANIFEST_SOURCE = DATA_ROOT / 'manifests'

RUN_PRETRAIN = True
RUN_FINETUNE = True
PRETRAIN_EPOCHS = None   # None means use config, currently 100 for ViT DINO.
FINETUNE_EPOCHS = None   # None means use config, currently 50.
NUM_WORKERS = 2

print('EXPERIMENT_ID:', EXPERIMENT_ID)
print('LABELLED_SOURCE:', LABELLED_SOURCE, LABELLED_SOURCE.exists())
print('UNLABELLED_SOURCE:', UNLABELLED_SOURCE, UNLABELLED_SOURCE.exists())
print('SYNTHETIC_SOURCE:', SYNTHETIC_SOURCE, SYNTHETIC_SOURCE.exists())
print('MANIFEST_SOURCE:', MANIFEST_SOURCE, MANIFEST_SOURCE.exists())
print('OUT:', OUT)


## 4. Link Data and Create Synthetic Manifest


In [ ]:
CLASSES = ['COVID', 'Lung_Opacity', 'Viral_Pneumonia', 'Normal']
CLASS_TO_LABEL = {'COVID': 0, 'Lung_Opacity': 1, 'Viral_Pneumonia': 2, 'Normal': 3}
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def list_images(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXTS)

def class_image_dir(root, cls):
    croot = Path(root) / cls
    return croot / 'images' if (croot / 'images').exists() else croot

def replace_path(target: Path, source: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() or target.is_symlink():
        if target.is_symlink() or target.is_file():
            target.unlink()
        else:
            shutil.rmtree(target)
    if source.exists():
        os.symlink(source, target, target_is_directory=source.is_dir())
        print('Linked', target, '->', source)
    else:
        raise FileNotFoundError(f'Missing source: {source}')

def write_synthetic_manifest(synthetic_root: Path, manifest_path: Path):
    rows = []
    for cls in CLASSES:
        files = list_images(class_image_dir(synthetic_root, cls))
        print(cls, len(files))
        for path in files:
            rel = Path('data/processed/synthetic_dcgan') / cls / 'images' / path.name
            rows.append({
                'image_path': str(rel),
                'class_name': cls,
                'label': CLASS_TO_LABEL[cls],
                'source': 'stage1_synthesis',
                'generator': 'DCGAN',
            })
    if not rows:
        raise ValueError(f'No synthetic images found at {synthetic_root}')
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with manifest_path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['image_path', 'class_name', 'label', 'source', 'generator'])
        writer.writeheader()
        writer.writerows(rows)
    print('Wrote synthetic manifest:', manifest_path, 'rows=', len(rows))

os.chdir(REPO_ROOT)
print('Working directory:', Path.cwd())
replace_path(REPO_ROOT / 'data/processed/labelled_4232', LABELLED_SOURCE)
replace_path(REPO_ROOT / 'data/processed/unlabelled_16934', UNLABELLED_SOURCE)
replace_path(REPO_ROOT / 'data/processed/synthetic_dcgan', SYNTHETIC_SOURCE)

manifest_dir = REPO_ROOT / 'data/manifests'
manifest_dir.mkdir(parents=True, exist_ok=True)
if MANIFEST_SOURCE.exists():
    for name in ['train.csv', 'val.csv', 'test.csv', 'labelled_all.csv', 'split_summary.json']:
        src = MANIFEST_SOURCE / name
        dst = manifest_dir / name
        if src.exists():
            shutil.copy2(src, dst)
            print('Copied manifest:', dst)
        elif dst.exists():
            print('Using repo manifest:', dst)
        else:
            raise FileNotFoundError(f'Missing manifest: {src}')
else:
    print('MANIFEST_SOURCE missing; using repo manifests if present:', MANIFEST_SOURCE)

write_synthetic_manifest(REPO_ROOT / 'data/processed/synthetic_dcgan', REPO_ROOT / 'data/manifests/synthetic_dcgan.csv')
!python scripts/check_experiment_inputs.py --synthetic-manifest data/manifests/synthetic_dcgan.csv


## 5. Pretrain ViT-S/16 with DINO on DCGAN Synthetic Images


In [ ]:
pretrain_cmd = [
    sys.executable, 'scripts/run_dino_vit.py',
    '--config', str(CONFIG_PATH),
    '--synthetic-manifest', 'data/manifests/synthetic_dcgan.csv',
    '--output-dir', str(OUT),
    '--resume-checkpoint', str(OUT / 'pretrain/checkpoints/last_dino_checkpoint.pth'),
    '--num-workers', str(NUM_WORKERS),
]
if PRETRAIN_EPOCHS is not None:
    pretrain_cmd += ['--epochs', str(PRETRAIN_EPOCHS)]
CKPT = OUT / 'pretrain/checkpoints/best_dino_teacher.pth'

if RUN_PRETRAIN:
    subprocess.run(pretrain_cmd, check=True)
else:
    print('Skipping pretraining')
print('Expected checkpoint:', CKPT, CKPT.exists())


## 6. Fine-Tune and Evaluate on Fixed Real Train/Val/Test Split


In [ ]:
finetune_cmd = [
    sys.executable, 'scripts/run_classification_vit.py',
    '--config', str(CONFIG_PATH),
    '--manifest-dir', 'data/manifests',
    '--output-dir', str(OUT),
    '--pretrained-checkpoint', str(OUT / 'pretrain/checkpoints/best_dino_teacher.pth'),
    '--num-workers', str(NUM_WORKERS),
]
if FINETUNE_EPOCHS is not None:
    finetune_cmd += ['--epochs', str(FINETUNE_EPOCHS)]
CKPT = OUT / 'pretrain/checkpoints/best_dino_teacher.pth'

if RUN_FINETUNE:
    if not CKPT.exists():
        raise FileNotFoundError(f'Missing pretrain checkpoint: {CKPT}')
    subprocess.run(finetune_cmd, check=True)
else:
    print('Skipping fine-tuning')


## 7. Display and Package Results


In [ ]:
metrics_path = OUT / 'metrics.json'
if metrics_path.exists():
    import pandas as pd, json
    display(pd.DataFrame([{**{'experiment_id': EXPERIMENT_ID}, **json.loads(metrics_path.read_text())}]))
else:
    print('metrics.json not found:', metrics_path)
!find "{OUT}" -maxdepth 4 -type f | sort
!cd /kaggle/working && zip -qr "{EXPERIMENT_ID}_results.zip" results/experiments_rerun_dcgan128/"{EXPERIMENT_ID}"
print('Result zip:', Path('/kaggle/working') / f'{EXPERIMENT_ID}_results.zip')
